In [1]:
!pip3 install -q -U bitsandbytes peft trl accelerate datasets transformers

In [2]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, GemmaTokenizer

In [3]:
os.environ['token']=userdata.get('Token')

In [4]:
modelId= "google/gemma-2b"

bnbConfig= BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # 4-bit normal floor
    bnb_4bit_compute_dtype=torch.float16,

)

In [7]:
tokenizer= AutoTokenizer.from_pretrained( modelId, token= os.environ['token'])

model= AutoModelForCausalLM.from_pretrained( modelId, quantization_config= bnbConfig, device_map={"":0}, token=os.environ['token'])


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [11]:
text= "Quote: Neural vocoder is better than "
device="cuda:0"

inputs= tokenizer(text, return_tensors="pt").to(device)
outputs= model.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Quote: Neural vocoder is better than 99% of the audio processing algorithms in the world.

Neural vocoder is a type of neural network that is used to generate audio. It


In [13]:
os.environ["WANDB_DISABLED"]= "false"

loraConfig= LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type= "CAUSAL_LM",
)

In [35]:
from datasets import load_dataset

data= load_dataset("flytech/python-codes-25k")
# data= data.map(lambda samples: tokenizer(samples["output"]), batched= True)

In [39]:
def formatting_function(ex):
  text= f"Query: {ex['instruction'][0]}\n Code: {ex['output'][0]}"
  return text

In [40]:
data['train']

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [ ]:
trainer= SFTTrainer(
    model= model,
    train_dataset= data["train"],
    args= transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=100,
        learning_rate=2e-4,
        fp16=False,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit",
    ),
    peft_config= loraConfig,
    formatting_func= formatting_function,
)

In [42]:
trainer.train()

Step,Training Loss
1,6.315983
2,6.309470
3,6.367422
4,6.081366
5,5.650905
6,5.003241
7,4.744185
8,3.829020
9,3.377866
10,2.925712


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69ab18f4-4236bc8f6f7065ef1586fe00;f38d5044-c75c-4035-bd05-570cdbc327a0)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in google/gemma-2b.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in google/gemma-2b - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=100, training_loss=1.0893844087421893, metrics={'train_runtime': 178.231, 'train_samples_per_second': 2.244, 'train_steps_per_second': 0.561, 'total_flos': 43020509184000.0, 'train_loss': 1.0893844087421893})

In [ ]:
text= "Query: Create a shopping list based on my inputs!"
device= "cuda:0"
inputs= tokenizer(text, return_tensors="pt").to(device)
outputs= model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# this vague output was caused due to under-training due to max-steps=100. changing max-steps to 500/1000 would probably work.